This is an approximation of my best model score to date - it drew heavily on the data loading and augmentation from previously posted authors. I then wanted to play around instead of going the xgboost route :-) This is similar to a student work i did using a GAT GNN with RNN - Adapted for this competition for the regression task! I also swapped out the rnn for the transformer idea that someone else used...It takes around 8hrs tho to train!!! The GAT GNN is slow...i am also not convinced my implementation of the graph is doing great, The most score seemed to be coming from the rdkit - but that is hard to judge as when i trained longer the loss was better but my score was worse :-( The score was 0.057 - which put me then in 4th - since the TG extra data is gone the transformer seems to be struggling so who knows! I also tried with some extra enhanced data but as yet nothing better....But with 8hrs on GPU I am out of quota!!! Maybe this is interesting for someone...maybe also not!!!

In [ ]:
!pip install /kaggle/input/torch-geometric-2-6-1/torch_geometric-2.6.1-py3-none-any.whl

In [ ]:
# install RDKit for offline use
!pip install /kaggle/input/rdkit-install-whl/rdkit_wheel/rdkit_pypi-2022.9.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

In [ ]:
# ===================================================================
# CELL 1: Imports and Setup
# ===================================================================

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from torch_geometric import utils
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from rdkit import Chem
from rdkit.Chem import Descriptors
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Competition targets and weights
TARGETS = ['Tg', 'FFV', 'Tc', 'Density', 'Rg']


print("✅ Setup complete!")

In [ ]:
# In Kaggle
from transformers import AutoModelForMaskedLM, AutoTokenizer

model_path = "/kaggle/input/chembert"
model = AutoModelForMaskedLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
from transformers import AutoModelForMaskedLM, AutoTokenizer
import torch

# ✅ VERIFY MODEL IS LOADED
print("🔍 Model Verification:")
print(f"✅ Model type: {type(model).__name__}")
print(f"✅ Model config: {model.config.model_type}")
print(f"✅ Hidden size: {model.config.hidden_size}")
print(f"✅ Vocab size: {tokenizer.vocab_size}")
print(f"✅ Model device: {next(model.parameters()).device}")

# ✅ TEST TOKENIZATION
test_smiles = "CCO"  # Ethanol
tokens = tokenizer.tokenize(test_smiles)
encoded = tokenizer.encode(test_smiles, return_tensors="pt")

print(f"\n🧪 Tokenization Test:")
print(f"✅ Input SMILES: {test_smiles}")
print(f"✅ Tokens: {tokens}")
print(f"✅ Encoded shape: {encoded.shape}")

# ✅ TEST MODEL FORWARD PASS (FIXED)
with torch.no_grad():
    outputs = model(encoded)
    
print(f"\n🚀 Model Forward Pass:")
print(f"✅ Output type: {type(outputs)}")
print(f"✅ Available attributes: {[attr for attr in dir(outputs) if not attr.startswith('_')]}")

# Access the correct attributes for MaskedLMOutput
if hasattr(outputs, 'logits'):
    print(f"✅ Logits shape: {outputs.logits.shape}")
    print(f"✅ Logits type: {type(outputs.logits)}")

if hasattr(outputs, 'hidden_states'):
    print(f"✅ Hidden states: {outputs.hidden_states}")
    
if hasattr(outputs, 'attentions'):
    print(f"✅ Attentions: {outputs.attentions}")

# ✅ TEST PREDICTIONS
with torch.no_grad():
    predictions = torch.softmax(outputs.logits, dim=-1)
    print(f"✅ Predictions shape: {predictions.shape}")
    
    # Get top predictions for each token
    top_predictions = torch.topk(predictions, k=3, dim=-1)
    print(f"✅ Top 3 predictions shape: {top_predictions.values.shape}")

# ✅ TEST FILL-MASK PIPELINE (if supported)
print(f"\n🎯 Pipeline Test:")
try:
    from transformers import pipeline
    
    # Check if model supports fill-mask
    if hasattr(tokenizer, 'mask_token') and tokenizer.mask_token:
        fill_mask = pipeline('fill-mask', model=model, tokenizer=tokenizer)
        print(f"✅ Fill-mask pipeline created with mask token: {tokenizer.mask_token}")
        
        # Test masking
        masked_input = f"C{tokenizer.mask_token}O"
        results = fill_mask(masked_input)
        print(f"✅ Mask prediction test: {masked_input}")
        print(f"✅ Top prediction: {results[0] if results else 'No results'}")
        
    else:
        print("ℹ️  No mask token found - this model may not support fill-mask")
        print(f"ℹ️  Special tokens: {tokenizer.special_tokens_map}")
        
except Exception as e:
    print(f"⚠️  Pipeline test failed: {e}")

print(f"\n🎉 MODEL SUCCESSFULLY LOADED AND WORKING!")
print("✅ The AttributeError was just a structure issue - your model is fine!")

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# ===================================================================
# CELL 3: RDKit Molecular Descriptors
# ===================================================================

print("🔄 Computing RDKit molecular descriptors...")

def get_molecular_descriptors(max_autocorr=10):
    """Get molecular descriptors - auto-discovered from RDKit"""
    descriptor_list_all = []
    test_mol = Chem.MolFromSmiles('CCO')
    
    for name in dir(Descriptors):
        if not name.startswith('_'):
            try:
                func = getattr(Descriptors, name)
                if callable(func):
                    result = func(test_mol)
                    if isinstance(result, (int, float)) and not np.isnan(result):
                        descriptor_list_all.append((name, func))
            except:
                pass
    
    # Limit AUTOCORR2D descriptors
    autocorr_descriptors = [(name, func) for name, func in descriptor_list_all 
                           if name.startswith('AUTOCORR2D_')]
    autocorr_descriptors.sort(key=lambda x: int(x[0].split('_')[-1]))
    limited_autocorr = autocorr_descriptors[:max_autocorr]
    
    other_descriptors = [(name, func) for name, func in descriptor_list_all 
                        if not name.startswith('AUTOCORR2D_')]
    
    descriptor_list = limited_autocorr + other_descriptors
    feature_names = [name for name, _ in descriptor_list]
    
    print(f"RDKit descriptors: {len(descriptor_list)}")
    return descriptor_list, feature_names

def smiles_to_rdkit_features(smiles_list, descriptor_functions):
    """Convert SMILES to RDKit feature matrix"""
    features = []
    
    for smiles in tqdm(smiles_list, desc="RDKit features"):
        mol_features = []
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                mol_features = [np.nan] * len(descriptor_functions)
            else:
                for name, func in descriptor_functions:
                    try:
                        value = func(mol)
                        if np.isinf(value) or abs(value) > 1e10:
                            value = np.nan
                        mol_features.append(value)
                    except:
                        mol_features.append(np.nan)
        except:
            mol_features = [np.nan] * len(descriptor_functions)
        
        features.append(mol_features)
    
    return np.array(features, dtype=float)

def clean_rdkit_features(X):
    """Handle NaN/inf values and impute missing data"""
    X_clean = X.copy()
    X_clean[np.isinf(X_clean)] = np.nan
    
    # Median imputation
    for i in range(X_clean.shape[1]):
        col = X_clean[:, i]
        if np.isnan(col).any():
            X_clean[np.isnan(col), i] = np.nanmedian(col) if not np.isnan(np.nanmedian(col)) else 0
    
    return X_clean

# Get descriptor functions
descriptor_functions, feature_names = get_molecular_descriptors()

# Compute features
print("Computing training RDKit features...")
train_rdkit = smiles_to_rdkit_features(train_df['SMILES'].values, descriptor_functions)
train_rdkit = clean_rdkit_features(train_rdkit)

print("Computing test RDKit features...")
test_rdkit = smiles_to_rdkit_features(test_df['SMILES'].values, descriptor_functions)
test_rdkit = clean_rdkit_features(test_rdkit)

# Scale features
rdkit_scaler = StandardScaler()
train_rdkit_scaled = rdkit_scaler.fit_transform(train_rdkit)
test_rdkit_scaled = rdkit_scaler.transform(test_rdkit)

print(f"RDKit features shape: {train_rdkit_scaled.shape}")


In [ ]:
# ===================================================================
# CELL 5: Model Architecture - ChemBERTa + GRAN
# ===================================================================

print("Defining model architecture...")

class GraphAttention(nn.Module):
    def __init__(self, in_features, out_features, n_heads=4, dropout=0.1, alpha=0.2):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.n_heads = n_heads
        self.dropout = dropout
        self.alpha = alpha

        self.W = nn.Linear(in_features, n_heads * out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)

        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a.weight)

        self.leakyrelu = nn.LeakyReLU(self.alpha)
        self.dropout_layer = nn.Dropout(self.dropout)

    def forward(self, h, adj):
        batch_size = h.size(0)
        num_nodes = h.size(1)

        Wh = self.W(h)
        Wh = Wh.view(batch_size, num_nodes, self.n_heads, self.out_features)
        Wh = Wh.permute(0, 2, 1, 3)

        Wh_repeated_i = Wh.unsqueeze(3).expand(-1, -1, -1, num_nodes, -1)
        Wh_repeated_j = Wh.unsqueeze(2).expand(-1, -1, num_nodes, -1, -1)
        concat = torch.cat([Wh_repeated_i, Wh_repeated_j], dim=-1)

        e = self.leakyrelu(self.a(concat).squeeze(-1))

        zero_vec = -9e15 * torch.ones_like(e)
        adj = adj.unsqueeze(1)
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=-1)
        attention = self.dropout_layer(attention)

        h_prime = torch.matmul(attention, Wh)
        h_prime = h_prime.permute(0, 2, 1, 3).contiguous()
        h_prime = h_prime.view(batch_size, num_nodes, -1)

        return h_prime

class PolymerChemBERTaGRAN(nn.Module):
    def __init__(self, rdkit_features, chemberta_path='/kaggle/input/chembert', 
                 node_features=9, graph_hidden_dim=128, fusion_dim=256, dropout=0.1):
        super().__init__()
        
        # Load ChemBERTa
        print("Loading ChemBERTa...")
        self.tokenizer = AutoTokenizer.from_pretrained(chemberta_path)
        self.chemberta = AutoModel.from_pretrained(chemberta_path)
        chemberta_dim = self.chemberta.config.hidden_size  # 768
        
        # SMILES processing
        self.smiles_projection = nn.Sequential(
            nn.Linear(chemberta_dim, graph_hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(graph_hidden_dim),
            nn.Dropout(dropout)
        )
        
        # Graph processing (GRAN)
        self.node_embedding = nn.Linear(node_features, graph_hidden_dim)
        self.node_ln = nn.LayerNorm(graph_hidden_dim)
        
        n_heads = 4
        out_features_per_head = graph_hidden_dim // n_heads
        
        self.gat_layers = nn.ModuleList([
            GraphAttention(graph_hidden_dim, out_features_per_head, n_heads, dropout)
            for _ in range(2)
        ])
        self.gat_layer_norms = nn.ModuleList([
            nn.LayerNorm(graph_hidden_dim) for _ in range(2)
        ])
        
        self.graph_pooling = nn.Sequential(
            nn.Linear(graph_hidden_dim, graph_hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(graph_hidden_dim),
            nn.Dropout(dropout)
        )
        
        # RDKit processing
        self.rdkit_projection = nn.Sequential(
            nn.Linear(rdkit_features, graph_hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(graph_hidden_dim),
            nn.Dropout(dropout)
        )
        
        # Fusion layer
        total_features = graph_hidden_dim * 3  # SMILES + Graph + RDKit
        self.fusion_layer = nn.Sequential(
            nn.Linear(total_features, fusion_dim),
            nn.ReLU(),
            nn.LayerNorm(fusion_dim),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim),
            nn.ReLU(),
            nn.LayerNorm(fusion_dim),
            nn.Dropout(dropout)
        )
        
        # Property prediction heads
        self.property_heads = nn.ModuleDict({
            prop: nn.Sequential(
                nn.Linear(fusion_dim, graph_hidden_dim),
                nn.ReLU(),
                nn.LayerNorm(graph_hidden_dim),
                nn.Dropout(dropout),
                nn.Linear(graph_hidden_dim, 1)
            ) for prop in TARGETS
        })
        
        self.dropout = nn.Dropout(dropout)
    
    def process_smiles(self, smiles_list):
        encodings = self.tokenizer(smiles_list, padding=True, truncation=True, 
                                 max_length=512, return_tensors='pt')
        
        device = next(self.parameters()).device
        input_ids = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)
        
        outputs = self.chemberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]
        
        return self.smiles_projection(cls_output)
    
    def process_graph(self, node_features, adjacency_matrix):
        h = self.node_embedding(node_features)
        h = self.node_ln(h)
        
        for gat_layer, layer_norm in zip(self.gat_layers, self.gat_layer_norms):
            h_residual = h
            h = gat_layer(h, adjacency_matrix)
            h = F.elu(h + h_residual)
            h = layer_norm(h)
            h = self.dropout(h)
        
        pooled = torch.mean(h, dim=1)
        return self.graph_pooling(pooled)
    
    def forward(self, smiles_list, node_features, adjacency_matrix, rdkit_features):
        smiles_repr = self.process_smiles(smiles_list)
        graph_repr = self.process_graph(node_features, adjacency_matrix)
        rdkit_repr = self.rdkit_projection(rdkit_features)
        
        combined_repr = torch.cat([smiles_repr, graph_repr, rdkit_repr], dim=-1)
        fused_repr = self.fusion_layer(combined_repr)
        
        predictions = {}
        for prop_name, head in self.property_heads.items():
            predictions[prop_name] = head(fused_repr).squeeze(-1)
        
        return predictions, {
            'smiles_repr': smiles_repr,
            'graph_repr': graph_repr,
            'rdkit_repr': rdkit_repr,
            'fused_repr': fused_repr
        }

# Initialize model
rdkit_feature_dim = train_rdkit_scaled.shape[1]
model = PolymerChemBERTaGRAN(rdkit_features=rdkit_feature_dim)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print("✅ Model architecture complete!")

In [ ]:
# ===================================================================
# CELL 6: Data Processing Pipeline
# ===================================================================

print("🔄 Creating data processing pipeline...")

def pyg_to_adjacency_matrix(edge_index, num_nodes, add_self_loops=True):
    adj = torch.zeros((num_nodes, num_nodes), dtype=torch.float)
    if edge_index.size(1) > 0:
        adj[edge_index[0], edge_index[1]] = 1.0
    if add_self_loops:
        adj.fill_diagonal_(1.0)
    return adj

def process_single_molecule(smiles, max_nodes=100):
    try:
        if smiles.startswith('$'):
            smiles = smiles[1:]
        
        pyg_graph = utils.from_smiles(smiles)
        if pyg_graph is None or pyg_graph.x.size(0) > max_nodes:
            return None
        
        node_features = pyg_graph.x
        num_nodes = node_features.size(0)
        
        if num_nodes < max_nodes:
            padding = torch.zeros((max_nodes - num_nodes, node_features.size(1)))
            node_features = torch.cat([node_features, padding], dim=0)
        
        adj_matrix = pyg_to_adjacency_matrix(pyg_graph.edge_index, num_nodes)
        if num_nodes < max_nodes:
            padded_adj = torch.zeros((max_nodes, max_nodes))
            padded_adj[:num_nodes, :num_nodes] = adj_matrix
            adj_matrix = padded_adj
        
        return {
            'node_features': node_features,
            'adjacency_matrix': adj_matrix,
            'num_nodes': num_nodes
        }
    except:
        return None

def create_hybrid_dataset(df_aug, rdkit_features, is_train=True, max_nodes=100):
    processed_data = []
    
    for i, row in tqdm(df_aug.iterrows(), total=len(df_aug), desc='Processing molecules'):
        smiles = row['SMILES']
        
        graph_data = process_single_molecule(smiles, max_nodes)
        if graph_data is None:
            continue
        
        rdkit_feat = torch.tensor(rdkit_features[i], dtype=torch.float32)
        
        labels = None
        if is_train:
            labels = {}
            for prop in TARGETS:
                val = row[prop]
                if pd.notna(val):
                    labels[prop] = torch.tensor(float(val), dtype=torch.float32)
                else:
                    labels[prop] = torch.tensor(float('nan'), dtype=torch.float32)
        
        processed_data.append({
            'smiles': smiles,
            'node_features': graph_data['node_features'],
            'adjacency_matrix': graph_data['adjacency_matrix'],
            'rdkit_features': rdkit_feat,
            'labels': labels,
            'id': row.get('id', i)
        })
    
    print(f"Successfully processed {len(processed_data)} molecules")
    return processed_data

class HybridPolymerDataset(Dataset):
    def __init__(self, processed_data, is_train=True):
        self.data = processed_data
        self.is_train = is_train
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        if self.is_train:
            return (item['smiles'], item['node_features'], item['adjacency_matrix'], 
                   item['rdkit_features'], item['labels'])
        else:
            return (item['smiles'], item['node_features'], item['adjacency_matrix'], 
                   item['rdkit_features'], item['id'])

def collate_hybrid_train(batch):
    smiles_list, node_features, adj_matrices, rdkit_features, labels = zip(*batch)
    
    node_features_batch = torch.stack(node_features)
    adj_batch = torch.stack(adj_matrices)
    rdkit_batch = torch.stack(rdkit_features)
    
    labels_batch = {}
    for prop in TARGETS:
        prop_values = torch.stack([label_dict[prop] for label_dict in labels])
        labels_batch[prop] = prop_values
    
    return list(smiles_list), node_features_batch, adj_batch, rdkit_batch, labels_batch

def collate_hybrid_test(batch):
    smiles_list, node_features, adj_matrices, rdkit_features, ids = zip(*batch)
    
    node_features_batch = torch.stack(node_features)
    adj_batch = torch.stack(adj_matrices)
    rdkit_batch = torch.stack(rdkit_features)
    
    return list(smiles_list), node_features_batch, adj_batch, rdkit_batch, list(ids)

# Create train/validation split
print("Creating train/validation split...")
train_indices = list(range(len(train_df_aug)))
train_idx, val_idx = train_test_split(train_indices, test_size=0.2, random_state=42, shuffle=True)

train_split_df = train_df_aug.iloc[train_idx].reset_index(drop=True)
val_split_df = train_df_aug.iloc[val_idx].reset_index(drop=True)

train_rdkit_split = train_rdkit_aug[train_idx]
val_rdkit_split = train_rdkit_aug[val_idx]

print(f"Train split: {len(train_split_df)}, Val split: {len(val_split_df)}")

# Create datasets
print("Creating datasets...")
train_processed = create_hybrid_dataset(train_split_df, train_rdkit_split, is_train=True, max_nodes=200)
val_processed = create_hybrid_dataset(val_split_df, val_rdkit_split, is_train=True, max_nodes=200)
test_processed = create_hybrid_dataset(test_df_clean, test_rdkit_scaled, is_train=False, max_nodes=200)

# Create dataloaders
batch_size = 8  # Small batch for memory constraints
train_dataset = HybridPolymerDataset(train_processed, is_train=True)
val_dataset = HybridPolymerDataset(val_processed, is_train=True)
test_dataset = HybridPolymerDataset(test_processed, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                         collate_fn=collate_hybrid_train, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                       collate_fn=collate_hybrid_train, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                        collate_fn=collate_hybrid_test, num_workers=0)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
print("Data processing complete!")

In [ ]:
# 🧪 QUICK TEST MODE - Set to True for fast testing
QUICK_TEST = False  # Change to False for full training

if QUICK_TEST:
    print("🧪 QUICK TEST MODE - Using minimal data...")
    
    # Create mini dataloaders with much smaller data
    print("Creating mini train/val datasets...")
    
    # Take tiny subsets from existing processed data
    mini_train_data = train_processed[:20]  # Only 20 molecules
    mini_val_data = val_processed[:10]      # Only 10 molecules
    
    # Create mini datasets
    mini_train_dataset = HybridPolymerDataset(mini_train_data, is_train=True)
    mini_val_dataset = HybridPolymerDataset(mini_val_data, is_train=True)
    
    # Create mini loaders
    train_loader = DataLoader(mini_train_dataset, batch_size=4, shuffle=True, 
                             collate_fn=collate_hybrid_train, num_workers=0)
    val_loader = DataLoader(mini_val_dataset, batch_size=4, shuffle=False, 
                           collate_fn=collate_hybrid_train, num_workers=0)
    
    # Reduce training parameters
    num_epochs = 3  # Instead of 15
    patience = 2    # Instead of 5
    
    print(f"Mini train batches: {len(train_loader)}, Mini val batches: {len(val_loader)}")
    print("This should complete in ~2-3 minutes!")
else:
    print("🔄 FULL TRAINING MODE")
    num_epochs = 60
    patience = 10

print("🔄 Starting training...")


In [ ]:
# ===================================================================
# CELL 7: Training with Loss Tracking
# ===================================================================

print("Starting training...")

def compute_losses_and_representations(model, smiles_list, node_features, adj_matrices, 
                                     rdkit_features, labels_batch):
    """Compute losses and track representation norms"""
    
    # Move data to device
    node_features = node_features.to(device)
    adj_matrices = adj_matrices.to(device) 
    rdkit_features = rdkit_features.to(device)
    labels_device = {prop: labels_batch[prop].to(device) for prop in labels_batch}
    
    # Forward pass
    predictions, representations = model(smiles_list, node_features, adj_matrices, rdkit_features)
    
    # Compute losses
    losses = {}
    total_loss = 0
    valid_props = 0
    
    for prop in predictions:
        if prop in labels_device:
            mask = ~torch.isnan(labels_device[prop])
            if mask.sum() > 0:
                pred_valid = predictions[prop][mask]
                target_valid = labels_device[prop][mask]
                mae = F.l1_loss(pred_valid, target_valid)
                weight = PROPERTY_WEIGHTS.get(prop, 1.0)
                weighted_mae = weight * mae
                losses[f'{prop}_total'] = weighted_mae.item()
                total_loss += weighted_mae
                valid_props += 1
    
    if valid_props > 0:
        total_loss = total_loss / valid_props
        losses['total'] = total_loss.item()
    else:
        losses['total'] = 0.0
        total_loss = torch.tensor(0.0, requires_grad=True)
    
    # Track representation magnitudes
    losses['smiles_repr_norm'] = torch.norm(representations['smiles_repr']).item()
    losses['graph_repr_norm'] = torch.norm(representations['graph_repr']).item()
    losses['rdkit_repr_norm'] = torch.norm(representations['rdkit_repr']).item()
    
    return total_loss, losses

def train_one_epoch(model, train_loader, optimizer, epoch):
    model.train()
    epoch_losses = {
        'total': [], 'smiles_repr_norm': [], 'graph_repr_norm': [], 'rdkit_repr_norm': []
    }
    for prop in TARGETS:
        epoch_losses[f'{prop}_total'] = []
    
    for batch_idx, (smiles_list, node_features, adj_matrices, rdkit_features, labels_batch) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}")):
        
        total_loss, batch_losses = compute_losses_and_representations(
            model, smiles_list, node_features, adj_matrices, rdkit_features, labels_batch
        )
        
        if total_loss.item() > 0:
            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        # Store losses
        for key, value in batch_losses.items():
            if key in epoch_losses:
                epoch_losses[key].append(value)
    
    # Average losses
    avg_losses = {}
    for key, values in epoch_losses.items():
        avg_losses[key] = np.mean(values) if values else 0.0
    
    return avg_losses

def validate_model(model, val_loader):
    model.eval()
    val_losses = {
        'total': [], 'smiles_repr_norm': [], 'graph_repr_norm': [], 'rdkit_repr_norm': []
    }
    for prop in TARGETS:
        val_losses[f'{prop}_total'] = []
    
    with torch.no_grad():
        for smiles_list, node_features, adj_matrices, rdkit_features, labels_batch in tqdm(val_loader, desc="Validating"):
            
            total_loss, batch_losses = compute_losses_and_representations(
                model, smiles_list, node_features, adj_matrices, rdkit_features, labels_batch
            )
            
            for key, value in batch_losses.items():
                if key in val_losses:
                    val_losses[key].append(value)
    
    # Average losses
    avg_val_losses = {}
    for key, values in val_losses.items():
        avg_val_losses[key] = np.mean(values) if values else 0.0
    
    return avg_val_losses

# Training setup
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

num_epochs = 60
patience = 10
best_val_loss = float('inf')
patience_counter = 0

train_history = []
val_history = []

print(f"Starting training on {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training loop
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)
    
    # Training
    train_losses = train_one_epoch(model, train_loader, optimizer, epoch+1)
    train_history.append(train_losses)
    
    # Validation
    val_losses = validate_model(model, val_loader)
    val_history.append(val_losses)
    
    # Learning rate scheduling
    scheduler.step(val_losses['total'])
    current_lr = optimizer.param_groups[0]['lr']
    
    # Print epoch summary
    print(f"Train Loss: {train_losses['total']:.6f}")
    print(f"Val Loss: {val_losses['total']:.6f}")
    print(f"LR: {current_lr:.8f}")
    print(f"Repr Norms - SMILES: {train_losses['smiles_repr_norm']:.3f}, "
          f"Graph: {train_losses['graph_repr_norm']:.3f}, "
          f"RDKit: {train_losses['rdkit_repr_norm']:.3f}")
    
    # Early stopping
    if val_losses['total'] < best_val_loss:
        best_val_loss = val_losses['total']
        patience_counter = 0
        torch.save(model.state_dict(), 'best_hybrid_model.pt')
        print(f"✓ New best model saved! Val loss: {best_val_loss:.6f}")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
    
    if patience_counter >= patience:
        print(f"Early stopping triggered after {epoch+1} epochs")
        break

# Load best model
model.load_state_dict(torch.load('best_hybrid_model.pt'))
print(f"Training completed. Best val loss: {best_val_loss:.6f}")


In [ ]:
# ===================================================================
# CELL 8: Loss Visualization
# ===================================================================

print("🔄 Creating loss plots...")

def plot_training_losses(train_history, val_history):
    """Plot comprehensive training analysis"""
    
    epochs = range(1, len(train_history) + 1)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Hybrid Model Training Analysis', fontsize=16)
    
    # Plot 1: Total Loss
    axes[0, 0].plot(epochs, [h['total'] for h in train_history], 'b-', label='Train', alpha=0.8)
    axes[0, 0].plot(epochs, [h['total'] for h in val_history], 'r-', label='Val', alpha=0.8)
    axes[0, 0].set_title('Total Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Weighted MAE')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Representation Norms (Train)
    axes[0, 1].plot(epochs, [h['smiles_repr_norm'] for h in train_history], 'g-', label='SMILES (ChemBERTa)', alpha=0.8)
    axes[0, 1].plot(epochs, [h['graph_repr_norm'] for h in train_history], 'orange', label='Graph (GRAN)', alpha=0.8)
    axes[0, 1].plot(epochs, [h['rdkit_repr_norm'] for h in train_history], 'purple', label='RDKit', alpha=0.8)
    axes[0, 1].set_title('Representation Magnitudes (Train)')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('L2 Norm')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Property-specific losses (Train)
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    for i, (prop, color) in enumerate(zip(TARGETS, colors)):
        prop_key = f'{prop}_total'
        if prop_key in train_history[0]:
            axes[0, 2].plot(epochs, [h[prop_key] for h in train_history], 
                           color=color, alpha=0.7, label=prop)
    
    axes[0, 2].set_title('Property-Specific Losses (Train)')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Weighted MAE')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Plot 4: Representation Norms (Val)
    axes[1, 0].plot(epochs, [h['smiles_repr_norm'] for h in val_history], 'g-', label='SMILES (ChemBERTa)', alpha=0.8)
    axes[1, 0].plot(epochs, [h['graph_repr_norm'] for h in val_history], 'orange', label='Graph (GRAN)', alpha=0.8)
    axes[1, 0].plot(epochs, [h['rdkit_repr_norm'] for h in val_history], 'purple', label='RDKit', alpha=0.8)
    axes[1, 0].set_title('Representation Magnitudes (Val)')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('L2 Norm')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 5: Property losses (Val)
    for i, (prop, color) in enumerate(zip(TARGETS, colors)):
        prop_key = f'{prop}_total'
        if prop_key in val_history[0]:
            axes[1, 1].plot(epochs, [h[prop_key] for h in val_history], 
                           color=color, alpha=0.7, label=prop)
    
    axes[1, 1].set_title('Property-Specific Losses (Val)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Weighted MAE')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Plot 6: Modality Contribution Analysis
    # Compare representation magnitudes to understand which modality contributes most
    final_train = train_history[-1]
    final_val = val_history[-1]
    
    modalities = ['SMILES\n(ChemBERTa)', 'Graph\n(GRAN)', 'RDKit\n(Descriptors)']
    train_norms = [final_train['smiles_repr_norm'], final_train['graph_repr_norm'], final_train['rdkit_repr_norm']]
    val_norms = [final_val['smiles_repr_norm'], final_val['graph_repr_norm'], final_val['rdkit_repr_norm']]
    
    x = np.arange(len(modalities))
    width = 0.35
    
    axes[1, 2].bar(x - width/2, train_norms, width, label='Train', alpha=0.8)
    axes[1, 2].bar(x + width/2, val_norms, width, label='Val', alpha=0.8)
    axes[1, 2].set_title('Final Representation Magnitudes')
    axes[1, 2].set_ylabel('L2 Norm')
    axes[1, 2].set_xticks(x)
    axes[1, 2].set_xticklabels(modalities)
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print summary statistics
    print("\nTRAINING SUMMARY:")
    print(f"Final train loss: {train_history[-1]['total']:.6f}")
    print(f"Final val loss: {val_history[-1]['total']:.6f}")
    print(f"Best val loss: {min([h['total'] for h in val_history]):.6f}")
    
    print(f"\n🔬 MODALITY ANALYSIS:")
    print(f"SMILES (ChemBERTa) norm: {final_val['smiles_repr_norm']:.3f}")
    print(f"Graph (GRAN) norm: {final_val['graph_repr_norm']:.3f}")
    print(f"RDKit norm: {final_val['rdkit_repr_norm']:.3f}")
    
    # Determine dominant modality
    max_norm = max(final_val['smiles_repr_norm'], final_val['graph_repr_norm'], final_val['rdkit_repr_norm'])
    if max_norm == final_val['smiles_repr_norm']:
        dominant = "SMILES (ChemBERTa)"
    elif max_norm == final_val['graph_repr_norm']:
        dominant = "Graph (GRAN)"
    else:
        dominant = "RDKit"
    
    
    for prop in TARGETS:
        prop_key = f'{prop}_total'
        if prop_key in val_history[-1]:
            print(f"{prop}: {val_history[-1][prop_key]:.6f}")

# Create the plots
plot_training_losses(train_history, val_history)


In [ ]:
# ===================================================================
# CELL 9: Final Predictions & Submission
# ===================================================================


# Set model to evaluation mode
model.eval()

# Collect all predictions
all_predictions = {prop: [] for prop in TARGETS}
all_ids = []

print("Predicting on test set...")
with torch.no_grad():
    for batch_idx, (smiles_list, node_features, adj_matrices, rdkit_features, ids) in enumerate(tqdm(test_loader, desc="Predicting")):
        
        # Move to device
        node_features = node_features.to(device)
        adj_matrices = adj_matrices.to(device)
        rdkit_features = rdkit_features.to(device)
        
        # Forward pass
        predictions, representations = model(smiles_list, node_features, adj_matrices, rdkit_features)
        
        # Store predictions
        for prop in TARGETS:
            all_predictions[prop].extend(predictions[prop].cpu().numpy())
        
        all_ids.extend(ids)

# Create submission dataframe
print("Creating submission file...")
submission = pd.DataFrame({'id': all_ids})

for prop in TARGETS:
    submission[prop] = all_predictions[prop]

# Ensure correct order by id
submission = submission.sort_values('id').reset_index(drop=True)

# Save submission
submission.to_csv('submission.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(f"Test samples processed: {len(submission)}")

print(submission.head(10))

for prop in TARGETS:
    values = submission[prop].values
    print(f"{prop}: min={values.min():.4f}, max={values.max():.4f}, mean={values.mean():.4f}, std={values.std():.4f}")

print(f"• Architecture: ChemBERTa + GRAN + RDKit")
print(f"• Training data: {len(train_df):,} molecules (with augmentation: {len(train_df_aug):,})")
print(f"• Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"• Best validation loss: {min([h['total'] for h in val_history]):.6f}")
print(f"• Training epochs: {len(train_history)}")

print("• submission.csv - Final competition submission")
print("• best_hybrid_model.pt - Best model weights")
print("• training_analysis.png - Training loss plots")

